In [1]:
# Connecting to postgreSQL
import psycopg


conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="banking_practice",
    user="postgres",
    password="manoj"
)


cursor = conn.cursor()


print("Connected to PostgreSQL successfully!")

Connected to PostgreSQL successfully!


In [20]:
# Show every ACTIVE account together with the owning customer's full name and email. Only Active accounts should appear.

cursor.execute("""
    select a.account_id,
        a.account_type,
        c.first_name ||' '|| c.last_name as full_name,
        c.email
    from customers as c join accounts as a 
        on a.customer_id = c.customer_id
    where a.status = 'Active'
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

(100001, 'Checking', 'Krishna Rai', 'krishna.rai1@mailbank.com')
(100004, 'Fixed Deposit', 'Gita Pandey', 'gita.pandey3@mailbank.com')
(100007, 'Checking', 'Indira Gurung', 'indira.gurung5@mailbank.com')
(100009, 'Savings', 'Sabina Bhattarai', 'sabina.bhattarai6@mailbank.com')
(100013, 'Checking', 'Arjun Bhandari', 'arjun.bhandari9@mailbank.com')
(100014, 'Fixed Deposit', 'Sita Maharjan', 'sita.maharjan10@mailbank.com')
(100017, 'Checking', 'Bidya Pandey', 'bidya.pandey13@mailbank.com')
(100018, 'Checking', 'Pramod Malla', 'pramod.malla14@mailbank.com')
(100021, 'Checking', 'Gita Karki', 'gita.karki15@mailbank.com')
(100023, 'Recurring Deposit', 'Sagar Khadka', 'sagar.khadka17@mailbank.com')
(100025, 'Savings', 'Bina Basnet', 'bina.basnet18@mailbank.com')
(100029, 'Fixed Deposit', 'Meera Khadka', None)
(100030, 'Recurring Deposit', 'Anita Bhattarai', 'anita.bhattarai22@mailbank.com')
(100038, 'Savings', 'Sunita Koirala', 'sunita.koirala28@mailbank.com')
(100040, 'Checking', 'Gita Bhatt

In [21]:
# Find every customer who currently has NO account at all.

cursor.execute("""
    select
        c.customer_id,
        c.first_name ||' '|| c.last_name as full_name
    from customers as c 
    left join accounts as a 
        on a.customer_id = c.customer_id
    where a.account_id is null
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

(9999, 'Sabina Bhattarai')


In [22]:
# Find every account whose customer_id does not match any row in the customers table (orphaned accounts).

cursor.execute("""
    select a.account_id,
        a.customer_id,
        a.account_type
    from accounts as a 
    left join customers as c 
        on a.customer_id = c.customer_id
    where c.customer_id is null;
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

(100280, 99999, 'Savings')


In [23]:
# Produce one result set of every customer and every account regardless of whether a match exists on either side, and label each row as 'Matched', 'No Account' or 'Missing Customer'.

cursor.execute("""
    SELECT
        c.customer_id,
        c.first_name,
        c.last_name,
        a.account_id,
        a.account_type,
        CASE
            WHEN c.customer_id IS NOT NULL
                AND a.account_id IS NOT NULL
                THEN 'Matched'
            WHEN c.customer_id IS NOT NULL
                AND a.account_id IS NULL
                THEN 'No Account'
            WHEN c.customer_id IS NULL
                AND a.account_id IS NOT NULL
                THEN 'Missing Customer'
        END AS relationship_status
    FROM customers AS c
    FULL OUTER JOIN accounts AS a
        ON c.customer_id = a.customer_id;
""")

rows = cursor.fetchall()
print(rows)

[(1, 'Krishna', 'Rai', 100001, 'Checking', 'Matched'), (3, 'Gita', 'Pandey', 100003, 'Savings', 'Matched'), (3, 'Gita', 'Pandey', 100004, 'Fixed Deposit', 'Matched'), (3, 'Gita', 'Pandey', 100005, 'Checking', 'Matched'), (5, 'Indira', 'Gurung', 100007, 'Checking', 'Matched'), (6, 'Sabina', 'Bhattarai', 100009, 'Savings', 'Matched'), (7, 'Suresh', 'Tamang', 100010, 'Fixed Deposit', 'Matched'), (9, 'Arjun', 'Bhandari', 100013, 'Checking', 'Matched'), (10, 'Sita', 'Maharjan', 100014, 'Fixed Deposit', 'Matched'), (12, 'Indira', 'Lama', 100016, 'Savings', 'Matched'), (13, 'Bidya', 'Pandey', 100017, 'Checking', 'Matched'), (14, 'Pramod', 'Malla', 100018, 'Checking', 'Matched'), (15, 'Gita', 'Karki', 100021, 'Checking', 'Matched'), (16, 'Sunita', 'Dahal', 100022, 'Recurring Deposit', 'Matched'), (17, 'Sagar', 'Khadka', 100023, 'Recurring Deposit', 'Matched'), (18, 'Bina', 'Basnet', 100025, 'Savings', 'Matched'), (21, 'Meera', 'Khadka', 100028, 'Checking', 'Matched'), (21, 'Meera', 'Khadka', 1

In [24]:
# For every transaction, show the transaction id, amount, account type, branch and the owning customer's full name - a single query joining three tables.

cursor.execute("""
    select t.transaction_id,
        t.amount,
        a.account_type,
        a.branch,
        c.first_name ||' '|| c.last_name as full_name
    from customers as c join accounts as a
    on c.customer_id = a.customer_id
    join transactions as t 
    on a.account_id = t.account_id
""")

rows = cursor.fetchall()
print(rows)

[(7000011, Decimal('272.45'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000017, Decimal('46086.82'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000012, Decimal('9482.80'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000002, Decimal('19581.27'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000005, Decimal('62000.39'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000010, Decimal('22994.74'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000013, Decimal('51194.54'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000014, Decimal('61654.59'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000016, Decimal('70825.65'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000003, Decimal('57068.55'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000006, Decimal('1830.05'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000018, Decimal('63474.32'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000001, Decimal('53248.58'), 'Checking', 'Dharan North', 'Krishna Rai'), (7000008, Decimal('48054.08'

In [25]:
# Find the total balance held at each branch, ordered from highest to lowest.

cursor.execute("""
    select
        a.branch,
        sum(a.balance)as total_balance
    from accounts as a
    group by branch
    order by total_balance desc
""")

rows = cursor.fetchall()
print(rows)

[('Pokhara City', Decimal('10670073.98')), ('Butwal West', Decimal('10630192.58')), ('Itahari Plaza', Decimal('10564591.36')), ('Biratnagar East', Decimal('8847755.92')), ('Bhaktapur Central', Decimal('8549832.73')), ('Lalitpur', Decimal('8107557.86')), ('Dharan North', Decimal('6776927.72')), ('Kathmandu Main', Decimal('6764720.69'))]


In [26]:
# Find the TOP 5 branches by total balance, counting only Active accounts.

cursor.execute("""
    select
        a.branch,
        sum(a.balance)as total_balance
    from accounts as a
    where a.status = 'Active'
    group by branch
    order by total_balance desc
    limit 5
""")

rows = cursor.fetchall()
print(rows)

[('Bhaktapur Central', Decimal('8422854.28')), ('Biratnagar East', Decimal('8096685.98')), ('Itahari Plaza', Decimal('7736224.00')), ('Butwal West', Decimal('7719054.95')), ('Pokhara City', Decimal('7384398.09'))]


In [27]:
# Find account types where the average balance exceeds 50,000. Round the average to 2 decimal places.

cursor.execute("""
    select
        a.account_type,
        round(avg(a.balance), 2) as average_balance
    from accounts as a
    group by account_type
    having avg(a.balance) > 50000
""")

rows = cursor.fetchall()
print(rows)

[('Checking', Decimal('70698.04')), ('Fixed Deposit', Decimal('409733.72')), ('Savings', Decimal('115019.03')), ('Recurring Deposit', Decimal('356334.44'))]


In [28]:
# Count how many accounts each customer holds, and list only customers who hold more than 1 account.

cursor.execute("""
    select
        c.customer_id,
        c.first_name ||' ' || c.last_name as full_name,
        count(a.account_id) as no_of_accounts
    from customers as c join accounts as a
    on c.customer_id = a.customer_id
    group by c.customer_id, c.first_name, c.last_name
    having count(a.account_id) > 1
""")

rows = cursor.fetchall()
print(rows)

[(58, 'Ganesh KC', 3), (184, 'Sagar Shrestha', 2), (116, 'Rita Bhandari', 2), (70, 'Ravi Chaudhary', 2), (52, 'Priya Shrestha', 3), (162, 'Ashok Lama', 3), (84, 'Indira Joshi', 3), (170, 'Sagar Subedi', 2), (176, 'Milan Pandey', 3), (92, 'Poonam Regmi', 2), (101, 'Bina Pandey', 2), (115, 'Rajesh Chaudhary', 3), (59, 'Rabin Neupane', 3), (65, 'Meera Shrestha', 2), (200, 'Poonam Rana', 3), (73, 'Ram Chaudhary', 3), (121, 'Prakash Sharma', 2), (161, 'Naveen Neupane', 2), (119, 'Champa Shrestha', 2), (9, 'Arjun Bhandari', 2), (196, 'Anjali Shrestha', 2), (15, 'Gita Karki', 2), (79, 'Nabin Adhikari', 2), (26, 'Sita Adhikari', 2), (187, 'Sushila Koirala', 2), (77, 'Nabin Acharya', 2), (30, 'Gita Bhattarai', 3), (21, 'Meera Khadka', 2), (131, 'Mina Tamang', 2), (3, 'Gita Pandey', 3), (17, 'Sagar Khadka', 2), (104, 'Deepak Poudel', 2), (165, 'Anjali Tamang', 3), (179, 'Sunita Chaudhary', 2), (35, 'Kavita Basnet', 3), (174, 'Sarala Magar', 3), (45, 'Gita Gurung', 2), (107, 'Mina Rai', 3), (6, '

In [29]:
# Find the branch and account_type combination that has the single highest total transaction amount.

cursor.execute("""
    select
        a.branch,
        a.account_type,
        sum(t.amount) as total_transaction_amount
    from accounts as a join transactions as t
    on a.account_id = t.account_id
    group by a.branch, a.account_type
    order by total_transaction_amount desc
    limit 1;
""")

rows = cursor.fetchall()
print(rows)

[('Butwal West', 'Recurring Deposit', Decimal('7754589.46'))]


In [30]:
# Find every customer whose COMBINED account balance is greater than the overall average balance across all accounts.

cursor.execute("""
    select
        c.customer_id,
        c.first_name ||' '|| c.last_name as full_name,
        sum(a.balance) as combined_balance
    from customers as c join accounts as a
    on c.customer_id = a.customer_id
    group by c.customer_id, c.first_name, c.last_name
    having sum(a.balance) > (select avg(balance) from accounts)
""")

rows = cursor.fetchall()
print(rows)

[(8, 'Amit Chaudhary', Decimal('656901.46')), (87, 'Milan Subedi', Decimal('260928.95')), (116, 'Rita Bhandari', Decimal('358682.02')), (51, 'Amit Yadav', Decimal('767208.77')), (146, 'Yogesh Magar', Decimal('710413.82')), (70, 'Ravi Chaudhary', Decimal('1227569.86')), (52, 'Priya Shrestha', Decimal('1022241.87')), (162, 'Ashok Lama', Decimal('435104.59')), (132, 'Santosh Pandey', Decimal('771743.01')), (84, 'Indira Joshi', Decimal('635815.43')), (170, 'Sagar Subedi', Decimal('927250.52')), (192, 'Sanjay Malla', Decimal('381260.51')), (169, 'Kavita Ghimire', Decimal('513093.76')), (176, 'Milan Pandey', Decimal('1040916.63')), (115, 'Rajesh Chaudhary', Decimal('1737508.08')), (60, 'Ram Joshi', Decimal('645880.27')), (97, 'Indira Regmi', Decimal('289740.07')), (108, 'Sagar Maharjan', Decimal('369093.15')), (59, 'Rabin Neupane', Decimal('941683.14')), (124, 'Sunita Lama', Decimal('748102.26')), (200, 'Poonam Rana', Decimal('575665.80')), (73, 'Ram Chaudhary', Decimal('982674.83')), (103, 

In [31]:
# Find accounts whose balance is above the average balance of their own account_type (correlated subquery).

cursor.execute("""
    select a.account_id,
    a.account_type,
    a.balance from accounts as a
    where a.balance > (
        select round(avg(a2.balance),2) from accounts as a2
        where a2.account_type = a.account_type
    );
""")

rows = cursor.fetchall()
print(rows)

[(100001, 'Checking', Decimal('179640.10')), (100005, 'Checking', Decimal('136843.80')), (100016, 'Savings', Decimal('150476.28')), (100018, 'Checking', Decimal('122447.42')), (100028, 'Checking', Decimal('125547.65')), (100034, 'Checking', Decimal('151798.60')), (100044, 'Checking', Decimal('74750.72')), (100054, 'Checking', Decimal('74652.52')), (100059, 'Savings', Decimal('166375.38')), (100060, 'Checking', Decimal('116883.59')), (100073, 'Checking', Decimal('126727.22')), (100100, 'Savings', Decimal('197701.62')), (100104, 'Checking', Decimal('114138.74')), (100116, 'Savings', Decimal('146407.51')), (100119, 'Checking', Decimal('109428.40')), (100123, 'Checking', Decimal('156007.95')), (100134, 'Savings', Decimal('192924.43')), (100142, 'Checking', Decimal('81443.55')), (100143, 'Checking', Decimal('109430.42')), (100146, 'Checking', Decimal('78498.60')), (100153, 'Checking', Decimal('100894.16')), (100158, 'Checking', Decimal('128958.82')), (100167, 'Checking', Decimal('139385.86'

In [32]:
# Using EXISTS, find every customer who has made at least one 'Withdrawal' transaction.

cursor.execute("""
    select 
        c.customer_id,
        c.first_name ||' '|| c.last_name as full_name
    from customers as c
    where exists (
        select 1
        from accounts as a
        join transactions as t
            on a.account_id = t.account_id
        where a.customer_id = c.customer_id
        and t.txn_type = 'Withdrawal'
    );
""")

rows = cursor.fetchall()
print(rows)

[(1, 'Krishna Rai'), (2, 'Anita Lama'), (3, 'Gita Pandey'), (4, 'Mina Pandey'), (5, 'Indira Gurung'), (6, 'Sabina Bhattarai'), (8, 'Amit Chaudhary'), (9, 'Arjun Bhandari'), (10, 'Sita Maharjan'), (11, 'Kavita KC'), (12, 'Indira Lama'), (13, 'Bidya Pandey'), (14, 'Pramod Malla'), (15, 'Gita Karki'), (17, 'Sagar Khadka'), (18, 'Bina Basnet'), (19, 'Ganesh Sharma'), (20, 'Vikram Subedi'), (21, 'Meera Khadka'), (22, 'Anita Bhattarai'), (23, 'Bidya Acharya'), (24, 'Poonam Magar'), (25, 'Sunita Dahal'), (26, 'Sita Adhikari'), (28, 'Sunita Koirala'), (29, 'Kalpana Ghimire'), (30, 'Gita Bhattarai'), (31, 'Kalpana Koirala'), (32, 'Devi Dahal'), (33, 'Suresh Karki'), (34, 'Poonam Bhattarai'), (35, 'Kavita Basnet'), (36, 'Sanjay Bhandari'), (37, 'Anita Adhikari'), (38, 'Vikram KC'), (39, 'Sita Neupane'), (40, 'Priya Lama'), (41, 'Kamala Karki'), (42, 'Ashok Shrestha'), (43, 'Bikash Acharya'), (45, 'Gita Gurung'), (46, 'Rajesh Adhikari'), (49, 'Sabina Neupane'), (51, 'Amit Yadav'), (52, 'Priya Shr

In [33]:
# Using NOT EXISTS, find every account that has never had a single transaction.

cursor.execute("""
    select 
        a.account_id,
        a.customer_id,
        a.balance,
        a.account_type
    from accounts as a
    where not exists (
        select 1
        from transactions as t
        where a.account_id = t.account_id
    );
""")

rows = cursor.fetchall()
print(rows)

[(100280, 99999, Decimal('15000.00'), 'Savings')]


In [34]:
# Using IN with a subquery, list customers who live in a city that has more than 3 customers.

cursor.execute("""
    select 
        c.customer_id,
        c.first_name ||' ' || c.last_name as full_name,
        c.city
    from customers as c
    where city in (
        select city from customers
        where city is not null
        group by city
        having count(*) > 3
    );
""")

rows = cursor.fetchall()
print(rows)

[(1, 'Krishna Rai', 'Pokhara'), (2, 'Anita Lama', 'Biratnagar'), (3, 'Gita Pandey', 'Biratnagar'), (4, 'Mina Pandey', 'Bhaktapur'), (5, 'Indira Gurung', 'Butwal'), (6, 'Sabina Bhattarai', 'Pokhara'), (7, 'Suresh Tamang', 'Dhangadhi'), (8, 'Amit Chaudhary', 'Janakpur'), (9, 'Arjun Bhandari', 'Janakpur'), (10, 'Sita Maharjan', 'Nepalgunj'), (12, 'Indira Lama', 'Biratnagar'), (13, 'Bidya Pandey', 'Bharatpur'), (14, 'Pramod Malla', 'Damak'), (15, 'Gita Karki', 'Butwal'), (16, 'Sunita Dahal', 'Nepalgunj'), (17, 'Sagar Khadka', 'Pokhara'), (18, 'Bina Basnet', 'Bhaktapur'), (19, 'Ganesh Sharma', 'Janakpur'), (20, 'Vikram Subedi', 'Nepalgunj'), (21, 'Meera Khadka', 'Dhangadhi'), (22, 'Anita Bhattarai', 'Butwal'), (23, 'Bidya Acharya', 'Dhangadhi'), (24, 'Poonam Magar', 'Lalitpur'), (25, 'Sunita Dahal', 'Kathmandu'), (26, 'Sita Adhikari', 'Hetauda'), (27, 'Gita KC', 'Itahari'), (28, 'Sunita Koirala', 'Dhangadhi'), (29, 'Kalpana Ghimire', 'Kathmandu'), (30, 'Gita Bhattarai', 'Kathmandu'), (31, '

In [35]:
# Using a subquery in the FROM clause (inline view), compute the number of accounts and average balance per branch, then keep only branches with more than 5 accounts.

cursor.execute("""
    select 
        branch,
        account_count,
        avg_balance
    from (
        select 
            branch,
            count(*) as account_count,
            round(avg(balance), 2) as avg_balance
        from accounts
        group by branch
    ) as branch_summary
    where account_count > 5;
""")

rows = cursor.fetchall()
print(rows)

[('Biratnagar East', 36, Decimal('245771.00')), ('Lalitpur', 27, Decimal('300279.92')), ('Pokhara City', 41, Decimal('260245.71')), ('Itahari Plaza', 39, Decimal('270886.96')), ('Butwal West', 35, Decimal('303719.79')), ('Kathmandu Main', 36, Decimal('187908.91')), ('Bhaktapur Central', 38, Decimal('224995.60')), ('Dharan North', 28, Decimal('242033.13'))]


In [36]:
# Combine the customer ids that hold a Savings account with the customer ids that hold a Checking account into ONE de-duplicated list, using UNION.

cursor.execute("""
    select 
        customer_id
    from accounts
    where account_type = 'Savings'

    union

    select 
        customer_id
    from accounts
    where account_type = 'Checking';
""")

rows = cursor.fetchall()
print(rows)

[(184,), (116,), (71,), (68,), (52,), (162,), (84,), (170,), (101,), (69,), (180,), (114,), (115,), (112,), (156,), (59,), (197,), (65,), (98,), (173,), (200,), (73,), (44,), (189,), (161,), (88,), (188,), (119,), (43,), (147,), (9,), (196,), (15,), (79,), (48,), (187,), (85,), (57,), (77,), (30,), (21,), (131,), (3,), (28,), (104,), (5,), (165,), (151,), (54,), (4,), (138,), (34,), (90,), (105,), (35,), (45,), (174,), (99999,), (107,), (6,), (134,), (39,), (89,), (36,), (31,), (102,), (14,), (167,), (109,), (13,), (155,), (133,), (111,), (199,), (75,), (128,), (99,), (142,), (46,), (53,), (32,), (183,), (38,), (136,), (150,), (193,), (12,), (137,), (78,), (191,), (25,), (141,), (122,), (186,), (33,), (1,), (106,), (18,), (110,), (178,), (145,), (55,), (129,), (143,), (58,)]


In [37]:
# Produce the same combined Savings/Checking customer list but KEEP duplicates (a customer with both types should appear twice), using UNION ALL.

cursor.execute("""
    select 
        customer_id
    from accounts
    where account_type = 'Savings'

    union all

    select 
        customer_id
    from accounts
    where account_type = 'Checking';
""")

rows = cursor.fetchall()
print(rows)

[(3,), (6,), (12,), (18,), (25,), (28,), (31,), (31,), (34,), (35,), (36,), (39,), (43,), (48,), (55,), (65,), (68,), (69,), (73,), (77,), (77,), (84,), (98,), (99,), (101,), (102,), (107,), (111,), (115,), (116,), (131,), (133,), (150,), (155,), (156,), (167,), (183,), (184,), (188,), (189,), (193,), (199,), (99999,), (4,), (6,), (32,), (71,), (84,), (90,), (99,), (119,), (136,), (137,), (143,), (145,), (170,), (174,), (191,), (1,), (3,), (5,), (9,), (13,), (14,), (15,), (21,), (25,), (30,), (31,), (33,), (38,), (44,), (45,), (46,), (52,), (53,), (54,), (55,), (57,), (58,), (59,), (73,), (75,), (78,), (79,), (85,), (88,), (89,), (104,), (105,), (106,), (107,), (107,), (109,), (109,), (110,), (112,), (114,), (119,), (122,), (128,), (129,), (131,), (134,), (138,), (141,), (142,), (147,), (151,), (161,), (161,), (162,), (162,), (165,), (173,), (178,), (180,), (184,), (186,), (187,), (187,), (196,), (197,), (200,), (200,)]


In [38]:
# Find customer ids that appear in BOTH the Savings list and the Checking list, using INTERSECT.

cursor.execute("""
    select 
        customer_id
    from accounts
    where account_type = 'Savings'

    intersect

    select 
        customer_id
    from accounts
    where account_type = 'Checking';
""")

rows = cursor.fetchall()
print(rows)

[(184,), (119,), (107,), (25,), (31,), (131,), (3,), (55,), (73,)]


In [39]:
# Find customer ids that have a Savings account but do NOT have a Fixed Deposit account, using EXCEPT.

cursor.execute("""
    select 
        customer_id
    from accounts
    where account_type = 'Savings'

    except

    select 
        customer_id
    from accounts
    where account_type = 'Fixed Deposite';
""")

rows = cursor.fetchall()
print(rows)

[(184,), (116,), (99,), (189,), (71,), (4,), (68,), (34,), (188,), (119,), (43,), (32,), (90,), (35,), (174,), (99999,), (183,), (136,), (150,), (107,), (6,), (193,), (84,), (48,), (12,), (170,), (137,), (39,), (191,), (101,), (77,), (69,), (36,), (25,), (31,), (115,), (102,), (167,), (131,), (3,), (28,), (156,), (155,), (133,), (65,), (111,), (18,), (199,), (145,), (55,), (143,), (98,), (73,)]


In [40]:
# Write a CTE that calculates each account's total transaction amount, then use it to list only accounts whose total exceeds 100,000.

cursor.execute("""
    with account_totals as(
        select 
            a.account_id,
            a.account_type,
            a.balance,
            sum(t.amount) as transaction_amount
        from accounts as a join transactions as t
            on a.account_id = t.account_id
        group by a.account_id
    )
    select * from account_totals 
    where transaction_amount > 100000
""")

rows = cursor.fetchall()
print(rows)

[(100186, 'Recurring Deposit', Decimal('484523.71'), Decimal('479032.31')), (100076, 'Savings', Decimal('0.00'), Decimal('111818.73')), (100027, 'Fixed Deposit', Decimal('282495.21'), Decimal('297966.94')), (100258, 'Savings', Decimal('1430.80'), Decimal('362687.83')), (100092, 'Fixed Deposit', Decimal('761647.59'), Decimal('436478.59')), (100220, 'Recurring Deposit', Decimal('297664.11'), Decimal('376627.17')), (100169, 'Recurring Deposit', Decimal('895036.41'), Decimal('241090.38')), (100162, 'Fixed Deposit', Decimal('278153.24'), Decimal('986444.16')), (100004, 'Fixed Deposit', Decimal('84465.72'), Decimal('273502.97')), (100001, 'Checking', Decimal('179640.10'), Decimal('748232.00')), (100274, 'Recurring Deposit', Decimal('340997.07'), Decimal('378971.88')), (100080, 'Recurring Deposit', Decimal('0.00'), Decimal('228850.51')), (100006, 'Savings', Decimal('218889.39'), Decimal('995061.38')), (100054, 'Checking', Decimal('74652.52'), Decimal('801726.79')), (100233, 'Fixed Deposit', D

In [41]:
# Write a CTE to find the single highest-balance account in EACH branch.

cursor.execute("""
    with highest_balance_account as(
        select 
            a.branch,
            max(a.balance) as highest_balance
        from accounts as a 
        group by a.branch
    )
    select 
        a.account_id,
        a.account_type,
        a.branch,
        a.balance
    from accounts as a 
    join highest_balance_account as h
        on a.branch = h.branch
        and a.balance = h.highest_balance 
""")

rows = cursor.fetchall()
print(rows)

[(100012, 'Recurring Deposit', 'Kathmandu Main', Decimal('860444.19')), (100083, 'Recurring Deposit', 'Lalitpur', Decimal('881912.75')), (100151, 'Fixed Deposit', 'Butwal West', Decimal('883643.08')), (100161, 'Fixed Deposit', 'Itahari Plaza', Decimal('874417.08')), (100169, 'Recurring Deposit', 'Dharan North', Decimal('895036.41')), (100172, 'Recurring Deposit', 'Biratnagar East', Decimal('837729.39')), (100202, 'Recurring Deposit', 'Bhaktapur Central', Decimal('834777.30')), (100246, 'Recurring Deposit', 'Pokhara City', Decimal('868069.30'))]


In [42]:
# Chain two CTEs together: the first totals Deposit transactions per account, the second joins that total to accounts and returns accounts whose total deposits exceed their current balance.

cursor.execute("""
   WITH deposit_total AS (
        SELECT
            account_id,
            sum(amount) as total_deposit
        from transactions 
        where txn_type = 'Deposit'
        group by account_id
    ),
    account_deposit as (
        select
            a.account_id,
            a.account_type,
            a.balance,
            d.total_deposit
        from accounts as a 
        join deposit_total as d
            on a.account_id = d.account_id
    )
    select
        account_id,
        account_type,
        balance,
        total_deposit
    from account_deposit
    where total_deposit > balance
""")

rows = cursor.fetchall()
print(rows)

[(100001, 'Checking', Decimal('179640.10'), Decimal('226966.87')), (100007, 'Checking', Decimal('4350.72'), Decimal('169799.90')), (100017, 'Checking', Decimal('19677.46'), Decimal('164433.41')), (100023, 'Recurring Deposit', Decimal('79217.61'), Decimal('93734.21')), (100025, 'Savings', Decimal('104592.30'), Decimal('165393.61')), (100028, 'Checking', Decimal('125547.65'), Decimal('157070.35')), (100029, 'Fixed Deposit', Decimal('76317.99'), Decimal('143433.86')), (100033, 'Savings', Decimal('0.00'), Decimal('26143.72')), (100034, 'Checking', Decimal('151798.60'), Decimal('153924.17')), (100040, 'Checking', Decimal('-3191.86'), Decimal('56575.65')), (100042, 'Recurring Deposit', Decimal('84605.14'), Decimal('87579.72')), (100044, 'Checking', Decimal('74750.72'), Decimal('100807.52')), (100045, 'Savings', Decimal('110313.98'), Decimal('228323.20')), (100047, 'Checking', Decimal('3311.94'), Decimal('184004.86')), (100051, 'Savings', Decimal('51575.47'), Decimal('205659.44')), (100052, '

In [43]:
# Create a VIEW named active_accounts_view exposing only Active accounts along with the owning customer's full name.

cursor.execute("""
    create or replace view active_accounts_view as
        select 
            a.account_id,
            c.customer_id,
            a.account_type,
            c.first_name ||' '|| c.last_name as full_name,
            a.status
        from accounts as a 
        join customers as c
            on a.customer_id = c.customer_id
        where a.status = 'Active'

""")

cursor.execute("""
    select * from active_accounts_view
""")

rows = cursor.fetchall()
print(rows)

[(100001, 1, 'Checking', 'Krishna Rai', 'Active'), (100004, 3, 'Fixed Deposit', 'Gita Pandey', 'Active'), (100007, 5, 'Checking', 'Indira Gurung', 'Active'), (100009, 6, 'Savings', 'Sabina Bhattarai', 'Active'), (100013, 9, 'Checking', 'Arjun Bhandari', 'Active'), (100014, 10, 'Fixed Deposit', 'Sita Maharjan', 'Active'), (100017, 13, 'Checking', 'Bidya Pandey', 'Active'), (100018, 14, 'Checking', 'Pramod Malla', 'Active'), (100021, 15, 'Checking', 'Gita Karki', 'Active'), (100023, 17, 'Recurring Deposit', 'Sagar Khadka', 'Active'), (100025, 18, 'Savings', 'Bina Basnet', 'Active'), (100029, 21, 'Fixed Deposit', 'Meera Khadka', 'Active'), (100030, 22, 'Recurring Deposit', 'Anita Bhattarai', 'Active'), (100038, 28, 'Savings', 'Sunita Koirala', 'Active'), (100040, 30, 'Checking', 'Gita Bhattarai', 'Active'), (100042, 30, 'Recurring Deposit', 'Gita Bhattarai', 'Active'), (100043, 31, 'Savings', 'Kalpana Koirala', 'Active'), (100044, 31, 'Checking', 'Kalpana Koirala', 'Active'), (100045, 31,

In [44]:
# Create a MATERIALIZED VIEW named branch_balance_summary that pre-aggregates total balance and account count per branch, and write the command to refresh it CONCURRENTLY.

cursor.execute("""
    create materialized view branch_balance_summary as
        select 
            a.branch,
            sum(a.balance) as total_balance,
            count(*) as account_count
        from accounts as a 
        group by a.branch

""")

cursor.execute("""
    create unique index branch_balance_summary_uq
        on branch_balance_summary(branch)
""")

cursor.execute("""
    refresh materialized view concurrently branch_balance_summary
""")

cursor.execute("""
    SELECT *
    FROM branch_balance_summary
""")

rows = cursor.fetchall()
print(rows)

[('Biratnagar East', Decimal('8847755.92'), 36), ('Lalitpur', Decimal('8107557.86'), 27), ('Pokhara City', Decimal('10670073.98'), 41), ('Itahari Plaza', Decimal('10564591.36'), 39), ('Butwal West', Decimal('10630192.58'), 35), ('Kathmandu Main', Decimal('6764720.69'), 36), ('Bhaktapur Central', Decimal('8549832.73'), 38), ('Dharan North', Decimal('6776927.72'), 28)]


In [45]:
# Using ROW_NUMBER(), return only the MOST RECENT transaction for every account.

cursor.execute("""
    with ranked_transaction as (
        select t.*, row_number() over(
            partition by account_id
            order by txn_date desc, txn_time desc
        ) as rn
        from transactions as t
    )
    select 
        transaction_id,
        account_id,
        txn_date,
        txn_time,
        txn_type,
        amount
    FROM ranked_transaction
    where rn = 1;
""")

rows = cursor.fetchall()
print(rows)

[(7000009, 100001, datetime.date(2026, 7, 18), datetime.time(7, 45), 'Withdrawal', Decimal('63454.54')), (7004614, 100002, datetime.date(2026, 9, 10), datetime.time(14, 30, 53, 522296), 'Fee', Decimal('500.00')), (7000055, 100003, datetime.date(2026, 8, 17), datetime.time(21, 15), 'POS Purchase', Decimal('1260.74')), (7000059, 100004, datetime.date(2026, 8, 24), datetime.time(13, 15), 'Deposit', Decimal('41139.79')), (7000079, 100005, datetime.date(2026, 6, 28), datetime.time(10, 30), 'Fee', Decimal('876.75')), (7004615, 100006, datetime.date(2026, 9, 10), datetime.time(14, 30, 53, 522296), 'Fee', Decimal('500.00')), (7000125, 100007, datetime.date(2026, 8, 11), datetime.time(8, 30), 'Fee', Decimal('245.81')), (7004616, 100008, datetime.date(2026, 9, 10), datetime.time(14, 30, 53, 522296), 'Fee', Decimal('500.00')), (7000178, 100009, datetime.date(2026, 8, 3), datetime.time(11, 30), 'Transfer In', Decimal('52520.66')), (7000185, 100010, datetime.date(2024, 8, 12), datetime.time(9, 0), 

In [46]:
# Using RANK(), rank customers by their total account balance so that tied balances share the same rank (with a gap afterward).

cursor.execute("""
    with customer_balance as (
        select 
            c.customer_id,
            c.first_name ||' '|| c.last_name as full_name,
            sum(a.balance) as total_balance
        from customers as c
        join accounts as a
            on c.customer_id = a.customer_id
        group by c.customer_id, c.first_name, c.last_name
    )
    select 
        customer_id,
        full_name,
        total_balance,
        rank() over(
            order by total_balance desc
        ) as balance_rank
    from customer_balance
    order by balance_rank
""")

rows = cursor.fetchall()
print(rows)

[(115, 'Rajesh Chaudhary', Decimal('1737508.08'), 1), (148, 'Bipin Ghimire', Decimal('1561901.71'), 2), (26, 'Sita Adhikari', Decimal('1561656.70'), 3), (165, 'Anjali Tamang', Decimal('1473065.84'), 4), (163, 'Meera Thapa', Decimal('1272404.68'), 5), (70, 'Ravi Chaudhary', Decimal('1227569.86'), 6), (121, 'Prakash Sharma', Decimal('1170984.52'), 7), (174, 'Sarala Magar', Decimal('1132024.52'), 8), (35, 'Kavita Basnet', Decimal('1067901.31'), 9), (176, 'Milan Pandey', Decimal('1040916.63'), 10), (52, 'Priya Shrestha', Decimal('1022241.87'), 11), (73, 'Ram Chaudhary', Decimal('982674.83'), 12), (109, 'Kamala Sharma', Decimal('978255.93'), 13), (59, 'Rabin Neupane', Decimal('941683.14'), 14), (170, 'Sagar Subedi', Decimal('927250.52'), 15), (9, 'Arjun Bhandari', Decimal('894499.60'), 16), (117, 'Sunita Khadka', Decimal('884507.20'), 17), (123, 'Vikram Sharma', Decimal('837729.39'), 18), (164, 'Nisha Lama', Decimal('835663.76'), 19), (94, 'Suresh Magar', Decimal('835561.46'), 20), (154, 'N

In [47]:
# Using DENSE_RANK(), rank branches by total transaction amount with NO gaps in the ranking numbers.

cursor.execute("""
    with branches_txn_amount as (
        select
            a.branch,
            sum(t.amount) as total_transaction_amount
        from accounts as a
        inner join transactions as t
            on a.account_id = t.account_id
        group by a.branch
    )
    select
        branch,
        total_transaction_amount,
        dense_rank() over(
            order by total_transaction_amount desc
        ) as amount_rank
    from branches_txn_amount
    order by amount_rank
""")

rows = cursor.fetchall()
print(rows)

[('Biratnagar East', Decimal('20650705.23'), 1), ('Pokhara City', Decimal('19003640.07'), 2), ('Butwal West', Decimal('18435855.22'), 3), ('Bhaktapur Central', Decimal('17953307.74'), 4), ('Itahari Plaza', Decimal('17054322.09'), 5), ('Dharan North', Decimal('16568290.23'), 6), ('Lalitpur', Decimal('16260425.11'), 7), ('Kathmandu Main', Decimal('14707337.57'), 8)]


In [48]:
# Using LAG(), show each transaction next to the amount of the PREVIOUS transaction on the same account, ordered by date.

cursor.execute("""
    select
        transaction_id,
        account_id,
        txn_date,
        txn_time,
        amount,
        lag(amount) over(
            partition by account_id
            order by txn_date,txn_time,transaction_id
        ) as previous_transaction
    from transactions
    order by account_id, txn_date, txn_time
""")

rows = cursor.fetchall()
print(rows)

[(7000011, 100001, datetime.date(2024, 2, 12), datetime.time(21, 0), Decimal('272.45'), None), (7000017, 100001, datetime.date(2024, 5, 5), datetime.time(12, 45), Decimal('46086.82'), Decimal('272.45')), (7000012, 100001, datetime.date(2024, 7, 10), datetime.time(11, 30), Decimal('9482.80'), Decimal('46086.82')), (7000002, 100001, datetime.date(2024, 9, 6), datetime.time(18, 0), Decimal('19581.27'), Decimal('9482.80')), (7000005, 100001, datetime.date(2025, 2, 13), datetime.time(13, 45), Decimal('62000.39'), Decimal('19581.27')), (7000010, 100001, datetime.date(2025, 2, 15), datetime.time(7, 45), Decimal('22994.74'), Decimal('62000.39')), (7000013, 100001, datetime.date(2025, 6, 6), datetime.time(6, 0), Decimal('51194.54'), Decimal('22994.74')), (7000014, 100001, datetime.date(2025, 7, 23), datetime.time(22, 0), Decimal('61654.59'), Decimal('51194.54')), (7000016, 100001, datetime.date(2025, 7, 27), datetime.time(11, 45), Decimal('70825.65'), Decimal('61654.59')), (7000003, 100001, dat

In [49]:
# Using LEAD(), show each transaction next to the amount of the NEXT transaction on the same account, and calculate the difference between them.

cursor.execute("""
    select
            transaction_id,
            account_id,
            txn_date,
            txn_time,
            amount,
            lead(amount) over(
                partition by account_id
                order by txn_date,txn_time,transaction_id
            ) as next_transaction,
            lead(amount) over(
                partition by account_id
                order by txn_date,txn_time,transaction_id
            ) - amount as difference
        from transactions
        order by account_id, txn_date, txn_time 
""")

rows = cursor.fetchall()
print(rows)

[(7000011, 100001, datetime.date(2024, 2, 12), datetime.time(21, 0), Decimal('272.45'), Decimal('46086.82'), Decimal('45814.37')), (7000017, 100001, datetime.date(2024, 5, 5), datetime.time(12, 45), Decimal('46086.82'), Decimal('9482.80'), Decimal('-36604.02')), (7000012, 100001, datetime.date(2024, 7, 10), datetime.time(11, 30), Decimal('9482.80'), Decimal('19581.27'), Decimal('10098.47')), (7000002, 100001, datetime.date(2024, 9, 6), datetime.time(18, 0), Decimal('19581.27'), Decimal('62000.39'), Decimal('42419.12')), (7000005, 100001, datetime.date(2025, 2, 13), datetime.time(13, 45), Decimal('62000.39'), Decimal('22994.74'), Decimal('-39005.65')), (7000010, 100001, datetime.date(2025, 2, 15), datetime.time(7, 45), Decimal('22994.74'), Decimal('51194.54'), Decimal('28199.80')), (7000013, 100001, datetime.date(2025, 6, 6), datetime.time(6, 0), Decimal('51194.54'), Decimal('61654.59'), Decimal('10460.05')), (7000014, 100001, datetime.date(2025, 7, 23), datetime.time(22, 0), Decimal('6

In [50]:
# Using a running-total window function, show every account's transactions in date order with a cumulative (running) amount.

cursor.execute("""
    select 
        transaction_id,
        account_id,
        txn_date,
        txn_time,
        amount,
        sum(amount) over(
            partition by account_id
            order by txn_date, txn_time, transaction_id
            rows between unbounded preceding and current row
        ) as running_total
    from transactions
    order by account_id, txn_date, txn_time
""")

rows = cursor.fetchall()
print(rows)

[(7000011, 100001, datetime.date(2024, 2, 12), datetime.time(21, 0), Decimal('272.45'), Decimal('272.45')), (7000017, 100001, datetime.date(2024, 5, 5), datetime.time(12, 45), Decimal('46086.82'), Decimal('46359.27')), (7000012, 100001, datetime.date(2024, 7, 10), datetime.time(11, 30), Decimal('9482.80'), Decimal('55842.07')), (7000002, 100001, datetime.date(2024, 9, 6), datetime.time(18, 0), Decimal('19581.27'), Decimal('75423.34')), (7000005, 100001, datetime.date(2025, 2, 13), datetime.time(13, 45), Decimal('62000.39'), Decimal('137423.73')), (7000010, 100001, datetime.date(2025, 2, 15), datetime.time(7, 45), Decimal('22994.74'), Decimal('160418.47')), (7000013, 100001, datetime.date(2025, 6, 6), datetime.time(6, 0), Decimal('51194.54'), Decimal('211613.01')), (7000014, 100001, datetime.date(2025, 7, 23), datetime.time(22, 0), Decimal('61654.59'), Decimal('273267.60')), (7000016, 100001, datetime.date(2025, 7, 27), datetime.time(11, 45), Decimal('70825.65'), Decimal('344093.25')), 

In [51]:
# Find duplicate customer records - customers who share the exact same first_name, last_name and dob.

cursor.execute("""
    select
        first_name,
        last_name,
        dob,
        count(*) as duplicated_account
    from customers
    group by first_name, last_name, dob
    having count(*) > 1;
""")

rows = cursor.fetchall()
print(rows)

[('Sabina', 'Bhattarai', datetime.date(1976, 11, 24), 2)]


In [52]:
# Write ONE query that finds customers missing a city or email (NULL or blank), and a SECOND query that finds orphaned accounts (customer_id with no matching customer row).

cursor.execute("""
    select 
        customer_id,
        first_name,
        last_name,
        city,
        email
    from customers 
    where city is null or trim(city) = '' 
    or email is null or trim(email) = ''
""")

rows = cursor.fetchall()
print(rows)

[(11, 'Kavita', 'KC', None, 'kavita.kc11@mailbank.com'), (21, 'Meera', 'Khadka', 'Dhangadhi', None)]


In [53]:
# SECOND query that finds orphaned accounts (customer_id with no matching customer row).

cursor.execute("""
    select 
        a.account_id,
        a.customer_id,
        a.account_type,
        a.balance
    from accounts as a
    left join customers as c
        on a.customer_id = c.customer_id
    where c.customer_id is null;
""")

rows = cursor.fetchall()
print(rows)

[(100280, 99999, 'Savings', Decimal('15000.00'))]


In [54]:
# Using CASE WHEN, bucket every Active account into 'Low' (< 10,000), 'Medium' (10,000-100,000) or 'High' (> 100,000), then count accounts in each bucket.

cursor.execute("""
    select
        case
            when balance < 10000 then 'Low'
            when balance between 10000 and 100000 then 'Medium'
            when balance >= 100000 then 'High'
        end as balance_bucket,
        count(*) as account_count
    from accounts
    where status = 'Active'
    group by
        case
            when balance < 10000 then 'Low'
            when balance between 10000 and 100000 then 'Medium'
            when balance >= 100000 then 'High'
        end
    order by balance_bucket;
""")

rows = cursor.fetchall()
print(rows)

[('High', 136), ('Low', 10), ('Medium', 47)]


In [55]:
try:
    cursor.execute("""
        BEGIN;

        CREATE TEMP TABLE fee_accounts AS
        SELECT
            account_id,
            balance AS old_balance,
            balance - 500 AS new_balance
        FROM accounts
        WHERE balance > 200000;

        UPDATE accounts AS a
        SET balance = f.new_balance
        FROM fee_accounts AS f
        WHERE a.account_id = f.account_id;

        INSERT INTO transactions (
            transaction_id,
            account_id,
            txn_date,
            txn_time,
            txn_type,
            channel,
            amount,
            currency,
            balance_after,
            merchant,
            description,
            is_flagged
        )
        SELECT
            COALESCE((SELECT MAX(transaction_id) FROM transactions), 0)
                + ROW_NUMBER() OVER (ORDER BY f.account_id),
            f.account_id,
            CURRENT_DATE,
            CURRENT_TIME,
            'Fee',
            'System',
            500,
            a.currency,
            f.new_balance,
            'Bank',
            'Maintenance Fee',
            false
        FROM fee_accounts AS f
        INNER JOIN accounts AS a
            ON a.account_id = f.account_id;

        COMMIT;
    """)

    print("Maintenance fee transaction completed successfully.")

except Exception as e:
    conn.rollback()
    print("Transaction failed. All changes have been rolled back.")
    print("Error:", e)

Transaction failed. All changes have been rolled back.
Error: relation "fee_accounts" already exists


In [56]:
# Using NTILE(4), split customers into 4 equal-sized income quartiles ordered by annual_income, then count how many customers fall in each quartile.

cursor.execute("""
    with income_quartiles as (
        select 
            customer_id,
            first_name ||' '|| last_name as full_name,
            annual_income,
            ntile(4) over(
                order by annual_income
            ) as income_quartile
        from customers
        where annual_income is not null
    )
    select 
        income_quartile,
        count(*) as customer_count
    from income_quartiles
    group by income_quartile
    order by income_quartile;
""")

rows = cursor.fetchall()
print(rows)

[(1, 51), (2, 50), (3, 50), (4, 50)]


In [57]:
# Find every customer with a credit_score below 500 who still holds at least one account with a balance above 200,000

cursor.execute("""
    SELECT
        c.customer_id,
        c.first_name || ' ' || c.last_name AS full_name,
        c.credit_score,
        a.account_id,
        a.balance
    FROM customers AS c
    JOIN accounts AS a
        ON c.customer_id = a.customer_id
    WHERE c.credit_score < 500
      AND a.balance > 200000
""")

rows = cursor.fetchall()
print(rows)

[(4, 'Mina Pandey', 492, 100006, Decimal('218889.39')), (11, 'Kavita KC', 440, 100015, Decimal('347594.80')), (15, 'Gita Karki', 353, 100020, Decimal('489268.32')), (19, 'Ganesh Sharma', 471, 100026, Decimal('503256.77')), (30, 'Gita Bhattarai', 328, 100041, Decimal('706959.36')), (35, 'Kavita Basnet', 319, 100049, Decimal('617390.44')), (35, 'Kavita Basnet', 319, 100050, Decimal('398935.40')), (40, 'Priya Lama', 412, 100056, Decimal('625189.45')), (63, 'Bidya Acharya', 494, 100088, Decimal('323041.08')), (67, 'Ganesh Basnet', 478, 100093, Decimal('409033.01')), (73, 'Ram Chaudhary', 431, 100101, Decimal('784973.21')), (74, 'Pramod Subedi', 496, 100103, Decimal('529592.75')), (82, 'Bipin Ghimire', 380, 100114, Decimal('464340.60')), (91, 'Kalpana Gurung', 477, 100125, Decimal('482070.99')), (94, 'Suresh Magar', 402, 100129, Decimal('835561.46')), (117, 'Sunita Khadka', 349, 100164, Decimal('884507.20')), (123, 'Vikram Sharma', 407, 100172, Decimal('837729.39')), (124, 'Sunita Lama', 47

In [58]:
# List every FLAGGED transaction (is_flagged = true) together with the owning customer's name, the branch and the channel used, ordered by amount descending.

cursor.execute("""
    SELECT
        t.transaction_id,
        t.amount,
        c.first_name || ' ' || c.last_name AS full_name,
        a.branch,
        t.channel
    FROM transactions AS t
    INNER JOIN accounts AS a
        ON t.account_id = a.account_id
    INNER JOIN customers AS c
        ON a.customer_id = c.customer_id
    WHERE t.is_flagged = true
    ORDER BY t.amount DESC;
""")

rows = cursor.fetchall()
print(rows)

[(7000679, Decimal('94983.27'), 'Gita Bhattarai', 'Bhaktapur Central', 'POS'), (7001864, Decimal('94872.56'), 'Nabin Adhikari', 'Pokhara City', 'Mobile App'), (7002380, Decimal('94816.61'), 'Ganesh Joshi', 'Bhaktapur Central', 'ATM'), (7003439, Decimal('94685.37'), 'Bipin Ghimire', 'Butwal West', 'Internet Banking'), (7002405, Decimal('94588.07'), 'Santosh Karki', 'Lalitpur', 'Mobile App'), (7003175, Decimal('94394.67'), 'Sneha Subedi', 'Biratnagar East', 'POS'), (7002412, Decimal('94280.43'), 'Mina Rai', 'Kathmandu Main', 'Branch'), (7003446, Decimal('94182.33'), 'Bipin Ghimire', 'Butwal West', 'Internet Banking'), (7004511, Decimal('94180.98'), 'Anjali Shrestha', 'Biratnagar East', 'IVR'), (7002182, Decimal('94178.14'), 'Indira Regmi', 'Itahari Plaza', 'Mobile App'), (7000774, Decimal('94129.03'), 'Kalpana Koirala', 'Lalitpur', 'Branch'), (7002753, Decimal('94082.21'), 'Sunita Khadka', 'Dharan North', 'IVR'), (7001603, Decimal('94051.07'), 'Ravi Chaudhary', 'Pokhara City', 'ATM'), (7

In [59]:
# Find every customer whose kyc_status is 'Expired' but who still has at least one 'Active' account (a compliance risk).

cursor.execute("""
    SELECT DISTINCT
        c.customer_id,
        c.first_name || ' ' || c.last_name AS full_name,
        c.kyc_status,
        a.account_id,
        a.status
    FROM customers AS c
    INNER JOIN accounts AS a
        ON c.customer_id = a.customer_id
    WHERE c.kyc_status = 'Expired'
    AND a.status = 'Active';
""")

rows = cursor.fetchall()
print(rows)

[(180, 'Naveen Subedi', 'Expired', 100251, 'Active'), (110, 'Bikash Gurung', 'Expired', 100152, 'Active'), (142, 'Sunita Malla', 'Expired', 100195, 'Active'), (200, 'Poonam Rana', 'Expired', 100279, 'Active'), (168, 'Deepak Karki', 'Expired', 100233, 'Active'), (159, 'Ravi Pandey', 'Expired', 100216, 'Active'), (1, 'Krishna Rai', 'Expired', 100001, 'Active'), (26, 'Sita Adhikari', 'Expired', 100035, 'Active'), (146, 'Yogesh Magar', 'Expired', 100200, 'Active'), (124, 'Sunita Lama', 'Expired', 100173, 'Active'), (200, 'Poonam Rana', 'Expired', 100277, 'Active'), (98, 'Milan Neupane', 'Expired', 100133, 'Active'), (5, 'Indira Gurung', 'Expired', 100007, 'Active'), (99, 'Sarita Shrestha', 'Expired', 100135, 'Active'), (170, 'Sagar Subedi', 'Expired', 100236, 'Active'), (170, 'Sagar Subedi', 'Expired', 100235, 'Active'), (164, 'Nisha Lama', 'Expired', 100225, 'Active'), (172, 'Naveen Poudel', 'Expired', 100238, 'Active'), (13, 'Bidya Pandey', 'Expired', 100017, 'Active'), (153, 'Bishnu Mag

In [60]:
# Find joint accounts (is_joint_account = true) whose balance is above the AVERAGE balance of all accounts in their own branch (correlated subquery).

cursor.execute("""
    SELECT
        a.account_id,
        a.customer_id,
        a.branch,
        a.account_type,
        a.balance
    FROM accounts AS a
    WHERE a.is_joint_account = true
    AND a.balance > (
        SELECT AVG(a2.balance)
        FROM accounts AS a2
        WHERE a2.branch = a.branch
    );
""")

rows = cursor.fetchall()
print(rows)

[(100012, 9, 'Kathmandu Main', 'Recurring Deposit', Decimal('860444.19')), (100026, 19, 'Pokhara City', 'Fixed Deposit', Decimal('503256.77')), (100035, 26, 'Butwal West', 'Recurring Deposit', Decimal('789702.08')), (100036, 26, 'Bhaktapur Central', 'Recurring Deposit', Decimal('771954.62')), (100066, 49, 'Pokhara City', 'Recurring Deposit', Decimal('322381.59')), (100071, 52, 'Pokhara City', 'Fixed Deposit', Decimal('673445.38')), (100085, 60, 'Butwal West', 'Fixed Deposit', Decimal('645880.27')), (100086, 61, 'Itahari Plaza', 'Recurring Deposit', Decimal('295579.88')), (100088, 63, 'Itahari Plaza', 'Recurring Deposit', Decimal('323041.08')), (100092, 66, 'Lalitpur', 'Fixed Deposit', Decimal('761647.59')), (100101, 73, 'Itahari Plaza', 'Fixed Deposit', Decimal('784973.21')), (100103, 74, 'Dharan North', 'Recurring Deposit', Decimal('529592.75')), (100105, 76, 'Itahari Plaza', 'Fixed Deposit', Decimal('438931.70')), (100121, 87, 'Dharan North', 'Recurring Deposit', Decimal('260928.95')